In [1]:
import pandas as pd
from config import *
import glob
import os

In [4]:
files = glob.glob(f"{ROOT}/data/raw/asset_classes/*.csv")
filename= []
for file in files:
    filename.append(os.path.basename(file))

In [5]:
filename

['BTC-USD_Bitcoin_daily.csv',
 'VNQ_Real Estate (REIT)_daily.csv',
 'AGG_US Aggregate Bonds_daily.csv',
 'IWF_US Large Cap Growth_daily.csv',
 'IWD_US Large Cap Value_daily.csv',
 'IWM_US Small Cap Equity_daily.csv',
 'TLT_Long-Term Treasury_daily.csv',
 'RSP_Equal Weight S&P 500_daily.csv',
 'EEM_Emerging Markets Equity_daily.csv',
 'SHY_Short-Term Treasury_daily.csv',
 'EFA_International Developed Equity_daily.csv',
 'GLD_Gold_daily.csv',
 'USO_Oil_daily.csv',
 'SPY_US Large Cap Equity_daily.csv',
 'HYG_High Yield Bonds_daily.csv']

In [6]:
asset_mapping = {
    'BTC-USD_Bitcoin_daily.csv': 'bitcoin',
    'VNQ_Real Estate (REIT)_daily.csv': 'real_estate',
    'AGG_US Aggregate Bonds_daily.csv': 'us_agg_bonds',
    'IWF_US Large Cap Growth_daily.csv': 'large_cap_growth',
    'IWD_US Large Cap Value_daily.csv': 'large_cap_value',
    'IWM_US Small Cap Equity_daily.csv': 'small_cap',
    'TLT_Long-Term Treasury_daily.csv': 'long_term_treasury',
    'RSP_Equal Weight S&P 500_daily.csv': 'equal_weight_sp500',
    'EEM_Emerging Markets Equity_daily.csv': 'emerging_markets',
    'SHY_Short-Term Treasury_daily.csv': 'short_term_treasury',
    'EFA_International Developed Equity_daily.csv': 'intl_developed',
    'GLD_Gold_daily.csv': 'gold',
    'USO_Oil_daily.csv': 'oil',
    'SPY_US Large Cap Equity_daily.csv': 'us_large_cap',
    'HYG_High Yield Bonds_daily.csv': 'high_yield_bonds',
}

In [9]:
def asset_cleaning(file, col=None, start_year= None):
    
    df = pd.read_csv(file).drop(index=[0,1]).reset_index(drop=True)
    df.columns = df.columns.str.lower()
    df.rename(columns={
        df.columns[0]: 'date',
    }, inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    cols = df.columns[1:]
    df[cols] = df[cols].astype('float')
    df[col] = round((df['close'].pct_change() * 100), 3)

    if start_year is not None:
        df = df[df['date'].dt.year >= start_year]
    
    if col is not None:    
        df_clean = df[['date', col]]
        return df_clean

    return df


def asset_combining(files):
    df_clean = None
    
    for file in files:
        filename = os.path.basename(file)
        df = asset_cleaning(file, col=asset_mapping[filename])
        if df_clean is None:
            df_clean = df
        else:
            df_clean = pd.merge(df_clean, df, on='date', how='outer')
                
    return df_clean 

        

In [17]:
df_daily = asset_combining(files)
df_monthly = df_daily.set_index('date').resample('ME').first().reset_index()
df_quarter = df_daily.set_index('date').resample('QE').first().reset_index()

In [20]:
df_daily.to_csv(f"{ROOT}/data/processed/asset_class_returns_daily.csv", index=False)
df_monthly.to_csv(f"{ROOT}/data/processed/asset_class_returns_monthly.csv", index=False)
df_quarter.to_csv(f"{ROOT}/data/processed/asset_class_returns_quarterly.csv", index=False)